In [2]:

import os
import shutil
from pathlib import Path
import pandas as pd
import re

# === Parâmetros principais ===
ORDER = 1  # 1 para 1ª ordem, 2 para 2ª ordem
IMAGES_DIR = Path("./da_artifacts")  # onde estão as imagens originais
EXT = ".png"  # extensão dos arquivos

# === Caminhos automáticos ===
LABELS_CSV = IMAGES_DIR / f"user_topic_cluster_labels_umap_order{ORDER}.csv"
OUT_DIR = Path(f"./clusters_{ORDER}order")

print(f"Usando arquivo de labels: {LABELS_CSV}")
print(f"Pasta de saída: {OUT_DIR}")

# === Função auxiliar ===
def build_image_name(user_id: str, topic: str, order: int, ext: str = ".png") -> str:
    topic_safe = re.sub(r"[^A-Za-z0-9_\-]+", "_", str(topic))
    return f"user_{user_id}_topic_{topic_safe}_order{order}_fixed{ext}"

# === Leitura do CSV ===
df = pd.read_csv(LABELS_CSV)
print(f"Registros carregados: {len(df):,}")

# === Criar pastas de saída ===
OUT_DIR.mkdir(parents=True, exist_ok=True)
clusters = sorted(df["cluster"].unique().tolist())
for c in clusters:
    (OUT_DIR / f"cluster_{c}").mkdir(parents=True, exist_ok=True)
    cluster_src = IMAGES_DIR / f"cluster_{c}_mean_heatmap_umap_order{ORDER}.png"  # cluster_0_mean_heatmap_umap_order1
    cluster_dst = OUT_DIR / f"cluster_{c}" / cluster_src.name
    if cluster_src.exists():
        shutil.copy2(cluster_src, cluster_dst)
    else:
        print(f"⚠️ Arquivo de heatmap do cluster não encontrado: {cluster_src}")
print(f"Clusters detectados: {clusters}")

# === Copiar arquivos ===
copied, missing = 0, []
for _, row in df.iterrows():
    uid = str(row["userId"])
    tpc = str(row["topic"])
    cluster = str(row["cluster"])

    img_name = build_image_name(uid, tpc, ORDER, EXT)
    src = IMAGES_DIR / img_name
    dst = OUT_DIR / f"cluster_{cluster}" / img_name

    if src.exists():
        shutil.copy2(src, dst)
        copied += 1
    else:
        missing.append(str(src))

print(f"\n✅ Imagens copiadas: {copied}")
if missing:
    print(f"⚠️ Imagens não encontradas: {len(missing)} (mostrando até 10)")
    for m in missing[:10]:
        print("  -", m)

print(f"\nArquivos organizados em: {OUT_DIR.resolve()}")


Usando arquivo de labels: da_artifacts/user_topic_cluster_labels_umap_order1.csv
Pasta de saída: clusters_1order
Registros carregados: 916
Clusters detectados: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]

✅ Imagens copiadas: 916

Arquivos organizados em: /Users/otacilio.maia/Desktop/studies/experimentosmestrado/helloworld/per_topic/clusters_1order
